# Install & Import

In [4]:
!pip install -q sentence-transformers faiss-gpu-cu12 rank_bm25

In [5]:
import pandas as pd
import numpy as np
import faiss

In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/models/johnsonhk88/cross-encoderms-marco-minilm-l-6-v2/transformers/v1/1/ms-marco-MiniLM-L-6-v2/config.json
/kaggle/input/models/johnsonhk88/cross-encoderms-marco-minilm-l-6-v2/transformers/v1/1/ms-marco-MiniLM-L-6-v2/README.md
/kaggle/input/models/johnsonhk88/cross-encoderms-marco-minilm-l-6-v2/transformers/v1/1/ms-marco-MiniLM-L-6-v2/tokenizer_config.json
/kaggle/input/models/johnsonhk88/cross-encoderms-marco-minilm-l-6-v2/transformers/v1/1/ms-marco-MiniLM-L-6-v2/pytorch_model.bin
/kaggle/input/models/johnsonhk88/cross-encoderms-marco-minilm-l-6-v2/transformers/v1/1/ms-marco-MiniLM-L-6-v2/special_tokens_map.json
/kaggle/input/models/johnsonhk88/cross-encoderms-marco-minilm-l-6-v2/transformers/v1/1/ms-marco-MiniLM-L-6-v2/.gitattributes
/kaggle/input/models/johnsonhk88/cross-encoderms-marco-minilm-l-6-v2/transformers/v1/1/ms-marco-MiniLM-L-6-v2/vocab.txt
/kaggle/input/models/johnsonhk88/cross-encoderms-marco-minilm-l-6-v2/transformers/v1/1/ms-marco-MiniLM-L-6-v2/flax_mode

# load dataset

In [7]:
train = pd.read_csv(dirname+"/train.csv")
val = pd.read_csv(dirname+"/val.csv")
test = pd.read_csv(dirname+"/test.csv")
laws_de = pd.read_csv(dirname+"/laws_de.csv")
court_considerations = pd.read_csv(dirname+"/court_considerations.csv")

In [8]:
print(f'''
"train":{train.shape},
"\nlaws_de:"{laws_de.shape},
"\ncourt_considerations:"{court_considerations.shape}''')


"train":(1139, 3),
"
laws_de:"(175933, 3),
"
court_considerations:"(2476315, 2)


In [9]:
train.head()

,query_id,query,gold_citations
0,train_0001,Die A AG betreibt seit den 1970er-Jahren auf d...,Art. 10a Abs. 1 USG;Art. 2 Abs. 1 UVPV;Art. 10...
1,train_0002,Die Alpha Consulting AG kann nun ihr Grundstüc...,Art. 975 ZGB;Art. 974 Abs. 2 ZGB;Art. 973 ZGB;...
2,train_0003,Das Kompetenzzentrum Völkerstrafrecht bei der ...,Art. 264m StGB
3,train_0004,Die Linzer Stadtbahn AG ('LiSA') ist die priva...,Art. 176 Abs. 1 IPRG;Art. 186 Abs. 1 IPRG;Art....
4,train_0005,Die Stadt Winterthur beschloss am 10. Februar ...,Art. 93 Abs. 1 BGG;Art. 89 Abs. 1 BGG;Art. 89 ...


In [10]:
laws_de.head()

,citation,text,title
0,Art. 1 112,Die Einwohnergemeinde Bern tritt der Schweizer...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
1,Art. 2 112,Die Einwohnergemeinde Bern wird ferner der Sch...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
2,Art. 3 Abs. 1 112,1 Falls die Schweizerische Eidgenossenschaft z...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
3,Art. 3 Abs. 2 112,2 Durch Anlage des neuen Verwaltungsgebäudes a...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
4,Art. 4 Abs. 1 112,1 Die Einwohnergemeinde Bern übernimmt im fern...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...


In [11]:
court_considerations.head()

,citation,text
0,BGE 139 I 2 E. 1.12.2011,betr. Verweigerung der Beiladung seien gutzuhe...
1,BGE 139 I 2 E. 2,Eventualiter sei die Rückweisung an die Vorins...
2,BGE 139 I 2 E. 5.1,"In der Sache ist vorweg zu prüfen, ob der Ents..."
3,BGE 139 I 2 E. 5.2,Art. 34 Abs. 1 BV gewährleistet in allgemeiner...
4,BGE 139 I 2 E. 5.3,Im vorliegenden Fall geht es nicht um die Gült...


laws_de is like law book

court_considerations is like case studies

| laws_de          | court_considerations |
| ---------------- | -------------------- |
| Rules            | Interpretations      |
| Short text       | Long text            |
| Structured       | Complex              |
| Easier retrieval | Harder retrieval     |


# Prepare Corpus

In [12]:
laws_de.columns

Index(['citation', 'text', 'title'], dtype='object')

In [13]:
corpus = laws_de.rename(columns={"citation":"id"})

In [14]:
corpus["full_text"] = corpus["title"]*3 + "" +corpus["text"] # More weight to title
docs = corpus["full_text"].tolist()
docs_id = corpus["id"].tolist()
title = corpus["title"].tolist()

# Model Initialization

In [15]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_Token")

In [16]:
from huggingface_hub import login
login(token=hf_token)

In [17]:
# cross Lingual
from sentence_transformers import SentenceTransformer, CrossEncoder
embed_model =SentenceTransformer("/kaggle/input/models/yethukmutt/bge-m3/transformers/m3/1/bge-m3") # best choice
reranker = CrossEncoder("/kaggle/input/models/johnsonhk88/cross-encoderms-marco-minilm-l-6-v2/transformers/v1/1/ms-marco-MiniLM-L-6-v2")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: /kaggle/input/models/johnsonhk88/cross-encoderms-marco-minilm-l-6-v2/transformers/v1/1/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
